# **Step2 제공파일_AI면접관 Agent_with Gradio**

## **1. 환경준비**

### (1) 구글 드라이브

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project_genai)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 라이브러리

In [ ]:
!pip install langchain_openai langchain_core langchain-community -q
!pip install -U langgraph
!pip install PyMuPDF python-docx gradio -q

### (3) OpenAI API Key 확인
* api_key.txt 파일에 다음의 키를 등록하세요.
    * OPENAI_API_KEY

In [ ]:
import os

def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/project_genai/'
# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [ ]:
print(os.environ['OPENAI_API_KEY'][:30])

## **2. App.py**

* 아래 코드에, Step1 혹은 고도화 된 Step2 파일 코드를 붙인다.
    * 라이브러리
    * 함수들과 그래프
* Gradio 코드는 그대로 사용하거나 일부 수정 가능

In [3]:
%%writefile app.py

####### 여러분의 함수와 클래스를 모드 여기에 붙여 넣읍시다. #######
## 1. 라이브러리 로딩 ---------------------------------------------
import pandas as pd
import numpy as np
import os
import openai
import random
import ast
import fitz
from docx import Document

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from typing import Annotated, Literal, Sequence, TypedDict, List, Dict
from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.output_parsers import CommaSeparatedListOutputParser
from langgraph.graph import StateGraph, END, START

import json
import random
## ---------------- 1단계 : 사전준비 ----------------------

# 1) 파일 입력 --------------------
def extract_text_from_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        doc = fitz.open(file_path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text
    elif ext == ".docx":
        doc = Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        raise ValueError("지원하지 않는 파일 형식입니다. PDF 또는 DOCX만 허용됩니다.")

# 2) State 선언 --------------------
class InterviewState(TypedDict):
    # 고정 정보
    resume_text: str
    resume_summary: str
    resume_keywords: List[Dict[str, str]]     # 기술명과 분류 함께 저장
    job_match_score: Dict[str, str]           # 직무 평가 결과
    core_strengths:List[str]                  # 핵심 강점 3가지
    career_path_suggestion: Dict[str, str]      # 커리어 성장 방향 예측
    resume_reliability: Dict[str, str]          # 이력서 신뢰도 평가
    question_strategy: Dict[str, Dict]

    # 인터뷰 로그
    current_question: str
    current_answer: str
    current_strategy: str
    conversation: List[Dict[str, str]]
    evaluation : List[Dict[str, str]]
    next_step : str
    used_strategy: List[str]

# 3) resume 분석 --------------------
def analyze_resume(state: InterviewState) -> InterviewState:

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.4)

    prompt = ChatPromptTemplate.from_template("""
    당신은 인사(HR) 전문가이자 커리어 분석가입니다.
    아래의 이력서 내용을 분석하여 다음 항목을 JSON 형식으로 출력하세요.

    분석 목표:
    1️. 핵심 요약 (STAR 방식)
        - Situation, Task, Action, Result 순으로 3문장 이내 요약
    2️. 기술 스킬 구조화
        - 각 기술에 대해 {{"기술": "...", "분류": "..."}} 형식으로 제시
        - 분류 예시: 언어, 프레임워크, 데이터베이스, 도구, 기타
    3️. 예상 직무 및 적합도 평가
        - 이력서 내용을 기반으로 예상 직무 1개 추정
        - 해당 직무에 대한 적합도(%)와 평가 요약 제공
        예시:
        "job_match_score": {{"예상 직무": "AI 개발자", "적합도": "88%", "평가": "데이터 처리 및 모델링 경험 우수"}}
    4️. 핵심 강점 3가지 도출
    5️. 커리어 성장 방향 예측 (Career Path Insight)
        - 이력서에 나타난 기술/경험을 바탕으로 향후 발전 가능성이 높은 직무나 산업 방향을 예측
        예시:
        {{"예상 성장 방향": "데이터 엔지니어 → AI 리서처", "설명": "데이터 전처리 및 모델링 경험이 풍부해 AI 연구개발로 성장 가능"}}
    6️. 이력서 신뢰도 평가 (Resume Authenticity Check)
        - 이력서 내 구체성, 근거, 수치, 문체 일관성을 고려해 신뢰도를 %로 평가
        - 간단한 설명 포함
        예시:
        {{"점수": "82%", "평가": "성과 근거가 명확하고 구체적이지만 일부 내용이 포괄적임"}}

    --- 이력서 내용 ---
    {resume_text}
    --------------------

    출력 형식 (JSON만 반환, 코드블록 금지):
    {{
      "resume_summary": "...",
      "resume_keywords": [{{"기술": "...", "분류": "..."}}],
      "job_match_score": {{"예상 직무": "...", "적합도": "...", "평가": "..."}},
      "core_strengths": ["...", "...", "..."],
      "career_path_suggestion": {{"예상 성장 방향": "...", "설명": "..."}},
      "resume_reliability": {{"점수": "...", "평가": "..."}}
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({"resume_text": state.get("resume_text", "")})
    result_text = response.content.strip()


    try:
        result = json.loads(result_text)
        resume_summary = result.get("resume_summary", "")
        resume_keywords = result.get("resume_keywords", [])
        job_match_score = result.get("job_match_score", {})
        core_strengths = result.get("core_strengths", [])
        career_path_suggestion = result.get("career_path_suggestion", {})
        resume_reliability = result.get("resume_reliability", {})
    except Exception:
        resume_summary = result_text
        resume_keywords = []
        job_match_score, core_strengths = {}, []
        career_path_suggestion, resume_reliability = {}, {}

    return {
        **state,
        "resume_summary": resume_summary,
        "resume_keywords": resume_keywords,
        "job_match_score": job_match_score,
        "core_strengths": core_strengths,
        "career_path_suggestion": career_path_suggestion,
        "resume_reliability": resume_reliability,
    }

# 4) 질문 전략 수립 --------------------

def generate_question_strategy(state: InterviewState) -> InterviewState:

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.5)

    used_strategy = state.get("current_strategy", "")
    all_strategies = ["경력 및 경험", "동기 및 커뮤니케이션", "논리적 사고 및 문제 해결",
                      "성장 가능성 및 학습 태도", "진위 검증 및 구체성 평가"]

    available_strategies = [s for s in all_strategies if s != used_strategy]

    if state.get("next_step", "") == "new_question":
        print(f"새로운 질문 전략 생성: 이전 전략 '{used_strategy}' 제외하고 진행합니다.")
        print(f"남은 전략 후보: {available_strategies}\n")

    prompt = ChatPromptTemplate.from_template("""
    당신은 전문 HR 면접관입니다.
    아래의 이력서 분석 결과를 참고하여, 지원자에게 적합한 맞춤형 면접 질문 전략을 수립하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 예상 직무 및 적합도 ---
    {job_match_score}

    --- 커리어 성장 방향 ---
    {career_path_suggestion}

    --- 이력서 신뢰도 ---
    {resume_reliability}

    고려할 전략 분야:
    {available_strategies}

    면접 질문 전략은 총 5가지 영역으로 구성합니다:
    1️. 경력 및 경험 (Career Experience)
        - 실무 경험, 프로젝트, 전공 활용 능력
    2️. 동기 및 커뮤니케이션 (Motivation & Collaboration)
        - 지원 동기, 팀워크, 협업 능력
    3️. 논리적 사고 및 문제 해결 (Logical Thinking)
        - 사고 구조, 문제 분석, 해결 과정
    4️. 성장 가능성 및 학습 태도 (Growth Potential)
        - 이력서에서 보이는 향후 성장 방향을 기반으로
    5️. 진위 검증 및 구체성 평가 (Authenticity & Detail)
        - 신뢰도 평가 결과를 기반으로 진위/구체성 확인 질문

    각 항목별로 아래 형식으로 출력하세요 (코드블록 금지):

    {{
      "경력 및 경험": {{
        "질문 방향": "지원자의 실무 경험과 전공 적용 능력을 검증",
        "예시 질문": ["예시1", "예시2", "예시3"]
      }},
      "동기 및 커뮤니케이션": {{
        "질문 방향": "협업 및 동기 부여 관련 질문",
        "예시 질문": ["예시1", "예시2"]
      }},
      "논리적 사고 및 문제 해결": {{
        "질문 방향": "논리적 사고력과 문제 해결 접근을 평가",
        "예시 질문": ["예시1", "예시2"]
      }},
      "성장 가능성 및 학습 태도": {{
        "질문 방향": "커리어 성장 방향을 바탕으로 성장 가능성 평가",
        "예시 질문": ["예시1", "예시2"]
      }},
      "진위 검증 및 구체성 평가": {{
        "질문 방향": "이력서 신뢰도와 구체성 검증",
        "예시 질문": ["예시1", "예시2"]
      }}
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "job_match_score": state.get("job_match_score", {}),
        "career_path_suggestion": state.get("career_path_suggestion", {}),
        "resume_reliability": state.get("resume_reliability", {}),
        "available_strategies": available_strategies
    })

    result_text = response.content.strip()

    try:
        strategy_dict = json.loads(result_text)
    except Exception:
        strategy_dict = {s: {} for s in available_strategies}

    if available_strategies:
        new_strategy = random.choice(available_strategies)
        state["current_strategy"] = new_strategy
    else:
        new_strategy = used_strategy
        print("사용 가능한 새로운 전략이 없습니다. 이전 전략 유지을 유지합니다.")

    return {
        **state,
        "question_strategy": strategy_dict,
        "current_strategy": new_strategy
    }

# 5) 1단계 하나로 묶기 --------------------

def preProcessing_Interview(file_path: str) -> InterviewState:
    resume_text = extract_text_from_file(file_path)

    state: InterviewState = {
        "resume_text": resume_text,
    }

    state = analyze_resume(state)

    state = generate_question_strategy(state)	
    
    available_categories = list(state.get("question_strategy", {}).keys())

    if not available_categories:
        print("질문 전략이 비어 있습니다. 기본 질문을 설정합니다.")
        selected_strategy = "경력 및 경험"
        selected_question = "이력서 기반 질문을 생성하지 못했습니다."
    else:
        selected_strategy = random.choice(available_categories)
        selected_section = state["question_strategy"].get(selected_strategy, {})
        example_questions = selected_section.get("예시 질문", [])

        if example_questions:
            selected_question = random.choice(example_questions)
        else:
            selected_question = f"{selected_strategy} 관련 질문을 생성하지 못했습니다."

    temp = [selected_strategy]		
        
    return {
        **state,
        "current_question": selected_question,
        "current_strategy": selected_strategy,
        "used_strategy": temp
    }

## ---------------- 2단계 : 면접 Agent ----------------------

# 1) 답변 입력 --------------------
def update_current_answer(state: InterviewState, user_answer: str) -> InterviewState:
    return {
        **state,
        "current_answer": user_answer.strip()
    }

# 2) 답변 평가 --------------------
def evaluate_answer(state: InterviewState) -> InterviewState:
    # print('evaluate_answer')
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    prompt_evaluate = ChatPromptTemplate.from_template("""
    당신은 면접관입니다. 아래의 질문과 지원자의 답변을 보고 평가하세요.

    평가 기준:
    1. 질문과의 연관성:
       - 상(우수): 질문의 핵심 의도에 정확히 부합하며, 전반적인 내용을 명확히 다룸
       - 중(보통): 질문과 관련은 있으나 핵심 포인트가 부족하거나 부분적으로 누락됨
       - 하(미흡): 질문과 관련이 약하거나 엉뚱한 내용을 중심으로 함

    2. 답변의 구체성:
       - 상(우수): 구체적인 사례, 수치, 프로젝트 내용 등으로 뒷받침됨
       - 중(보통): 사례나 근거는 있으나 구체성이 다소 부족함
       - 하(미흡): 모호하고 추상적인 답변

    --- 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    JSON 형식으로만 평가 결과를 작성하세요. 코드블록(```)은 사용하지 마세요.
    {{
      "질문과의 연관성": "상/중/하 중 하나",
      "답변의 구체성": "상/중/하 중 하나",
      "총평": "전반적인 평가 요약 (1~2문장)"
    }}


    """)

    #  실행
    chain = prompt_evaluate | llm
    response = chain.invoke({
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", "")
    })

    result_text = response.content.strip()

    # JSON 파싱
    try:
        evaluation = json.loads(result_text)
    except Exception:
        evaluation = {
            "질문과의 연관성": "평가 실패",
            "답변의 구체성": "평가 실패",
            "총평": result_text
        }

    # 대화 로그 및 평가 추가
    conversation = state.get("conversation", [])
    conversation.append({
        "role": "human",
        "content": state.get("current_answer", "")
    })
    conversation.append({
        "role": "ai",
        "content": f"평가 결과: {evaluation}"
    })

    evaluation_list = state.get("evaluation", [])
    evaluation_list.append({
        "question": state.get("current_question", ""),
        "answer": state.get("current_answer", ""),
        "question_strategy": state.get("current_strategy", ""), # question_strategy 추가
        "evaluation": evaluation
    })

    return {
        **state,
        "conversation": conversation,
        "evaluation": evaluation_list
    }

def reflect_on_evaluation(state: InterviewState) -> InterviewState:
    # print('reflect_on_evaluation')
    """LLM을 이용해 최근 평가가 적절한지 되돌아보는 Reflection 노드"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    last_eval = state.get("evaluation", [])[-1] if state.get("evaluation") else {}
    eval_fields = last_eval.get("evaluation", {})
    question = state.get("current_question", "")
    answer = state.get("current_answer", "")

    prompt = ChatPromptTemplate.from_template("""
    당신은 AI 면접 평가 품질 관리관입니다.

    아래의 질문, 답변, 기존 평가를 검토하여 평가가 적절한지 판단하세요.
    만약 평가가 부정확하거나 보완이 필요하다면 그 이유를 구체적으로 말하고,
    최종 판단을 "정상" 또는 "재평가 필요" 중 하나로 내리세요.

    --- 질문 ---
    {question}

    --- 지원자 답변 ---
    {answer}

    --- 기존 평가 ---
    {evaluation}

    JSON 형식으로만 응답하세요:
    {{
      "판단": "정상" 또는 "재평가 필요",
      "이유": "판단 근거를 1~2문장으로 서술"
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({
        "question": question,
        "answer": answer,
        "evaluation": json.dumps(eval_fields, ensure_ascii=False)
    })

    result_text = response.content.strip()

    try:
        reflection = json.loads(result_text)
    except Exception:
        reflection = {"판단": "재평가 필요", "이유": "LLM 응답 파싱 실패"}

    state["reflection_status"] = reflection.get("판단", "재평가 필요")
    state["reflection_feedback"] = reflection.get("이유", "")

    state["next_step"] = (
        "re_evaluate" if state["reflection_status"] == "재평가 필요"
        else "decide_next_step"
    )

    return state

def re_evaluate_answer(state: InterviewState) -> InterviewState:
    # print('re_evaluate_answer')
    """
    Reflection 결과가 '재평가 필요'일 때 실행되는 재평가 노드.
    기존 evaluate_answer()와 유사하지만, 재평가 이력을 남기고
    이전 평가를 참고하여 보완 평가를 수행한다.
    """
    re_eval_count = state.get("re_eval_count", 0) + 1
    state["re_eval_count"] = re_eval_count

#재평가 2회까지만
    if re_eval_count > 2:
        state["reflection_status"] = "정상"
        state["reflection_feedback"] = "재평가 한도를 초과하여 평가를 확정합니다."
        state["next_step"] = "decide_next_step"
        return state

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    last_eval = state.get("evaluation", [])[-1] if state.get("evaluation") else {}
    prev_evaluation = last_eval.get("evaluation", {})

    prompt = ChatPromptTemplate.from_template("""
    당신은 AI 면접관입니다. 아래는 이전 평가 결과와 새로운 답변입니다.
    이전 평가가 미흡했다고 판단되어 재평가를 수행합니다.

    이전 평가:
    {prev_evaluation}

    --- 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    JSON 형식으로만 재평가 결과를 작성하세요. 코드블록(```)은 사용하지 마세요.
    {{
      "질문과의 연관성": "상/중/하 중 하나",
      "답변의 구체성": "상/중/하 중 하나",
      "총평": "보완된 평가 요약 (1~2문장)"
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "prev_evaluation": json.dumps(prev_evaluation, ensure_ascii=False)
    })

    result_text = response.content.strip()

    # JSON 파싱
    try:
        new_evaluation = json.loads(result_text)
    except Exception:
        new_evaluation = {
            "질문과의 연관성": "평가 실패",
            "답변의 구체성": "평가 실패",
            "총평": result_text
        }

    # 평가 리스트에 새 평가 추가
    evaluation_list = state.get("evaluation", [])
    evaluation_list.append({
        "question": state.get("current_question", ""),
        "answer": state.get("current_answer", ""),
        "evaluation": new_evaluation,
        "re_evaluation": True  # 재평가 여부 표시
    })

    # 다음 단계는 다시 Reflection으로
    state["next_step"] = "reflect"

    return {
        **state,
        "evaluation": evaluation_list,
    }

# 3) 인터뷰 진행 검토 --------------------

def decide_next_step(state: InterviewState) -> InterviewState:
	evaluation_list = state.get("evaluation", [])
	question_count = len(evaluation_list)
   
	
	last_eval = evaluation_list[-1].get("evaluation", {})
	relation_score = last_eval.get("질문과의 연관성", "")
	detail_score = last_eval.get("답변의 구체성", "")
	# 기본값
	next_step = "additional_question"
	additional_count =state.get("additional_count", 0)
	# 조건 분기
	if question_count >= 5:
		# print("질문 횟수 5번 이상으로 종료")
		next_step = "end"
	elif "하" in (relation_score, detail_score):
		# print("\"하\"가 포함되어 새로운 질문 선택")
		next_step = "new_question"
	elif additional_count>=2:
		# print("심화질문 2번 이상으로 새로운 질문 선택")
		next_step = "new_question"
		additional_count=0
	else:
		if random.random()<=0.4:
			# print("랜덤으로 심화질문 질문 생성")
			next_step = 'additional_question'
		else:
			#print("랜덤으로 새로운 질문 선택")
			next_step = 'new_question'
			
	return {
		**state,
		"next_step": next_step,
	}

# 4) 질문 생성 --------------------
import shutil
from langchain_core.documents import Document

def generate_question(state: InterviewState) -> InterviewState:

    """
      질문 생성 (고도화 버전)
        - Chroma Vector DB에서 유사 질문 3개를 검색하여 LLM 프롬프트에 참고로 포함

    """

    # LLM 및 임베딩 모델 정의
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")

    # 유사 질문 생성 프롬프트
    sub_prompt = ChatPromptTemplate.from_template("""
    당신은 전문 면접관입니다. 아래 대화 맥락을 참고하여,
    현재 질문과 유사한 의도나 주제를 가진 면접 질문 3개를 생성하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 최근 질문 ---
    {current_question}

    --- 현재 질문 전략 ---
    {current_strategy}

    --- 지원자 답변 ---
    {current_answer}

    생성 규칙:
    1. 각 질문은 현재 질문과 **의도, 주제, 평가 관점**이 유사해야 합니다.
    2. 문장 구조나 표현은 다양하게 바꿔서 작성하세요. (동어 반복 금지)
    3. 한 문장으로 작성하고, 자연스럽고 명확한 한국어 질문 형태로 만드세요.
    4. 어조는 전문적이며 격식체를 유지합니다.
    5. 모든 질문은 30자 이내로 간결하게 작성하세요.
    6. 강조기호(** 등)나 이모티콘은 사용하지 마세요.

    출력 형식:
    1) 질문 1
    2) 질문 2
    3) 질문 3
    """)

    response = (sub_prompt | llm).invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "current_strategy": state.get("current_strategy","")

    })

    # 결과 파싱
    lines = [line.strip("123). ").strip() for line in response.content.split("\n") if line.strip()]
    similar_questions = [q for q in lines if q and len(q) > 3][:3]


    # 벡터 DB 생성
    docs = [Document(page_content=q) for q in similar_questions]
    vectorstore = Chroma.from_documents(docs, embedding, persist_directory=None)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 최종 질문 생성

    similar_docs = retriever.invoke(state.get("current_answer", ""))
    similar_texts = "\n".join([f"- {doc.page_content}" for doc in similar_docs])

    main_prompt = ChatPromptTemplate.from_template("""
    당신은 면접관입니다. 아래 정보를 기반으로 지원자에게 추가로 물어볼 질문을 생성하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 질문 전략 ---
    {question_strategy}

    --- 현재 질문 전략 ---
    {current_strategy}

    --- 이전 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    --- 답변 평가 ---
    {evaluation}

    --- 참고용 유사 질문 ---
    {similar_examples}

    생성 규칙:
    1. 지원자의 답변 내용을 기반으로 핵심 근거, 구체적 행동, 결과를 더 깊이 탐색하는 심화 질문을 생성합니다.
    2. 새로운 주제로 전환하지 말고, 반드시 이전 질문의 맥락을 유지하세요.
    3. 한 문장으로 자연스럽고 명확한 질문을 작성합니다.
    4. 어조는 전문적이며 격식체를 유지합니다.
    5. 질문은 30자 이내로 간결하게 작성하세요.
    6. 강조기호(** 등)나 이모티콘은 사용하지 마세요.

    """)


    chain = main_prompt | llm
    response = chain.invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "question_strategy": state.get("question_strategy", {}),
        "current_strategy" : state.get("current_strategy",""),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "evaluation": state.get("evaluation", []),
        "next_step": state.get("next_step",""),
        "similar_examples": similar_texts
    })

    next_question = response.content.strip()

    return {
        **state,
        "current_question": next_question,
        "current_answer": "",
    }

# 5) 인터뷰 피드백 보고서 --------------------
from collections import defaultdict

def summarize_interview(state: InterviewState) -> InterviewState:
    print("\n\n===== 인터뷰 피드백 보고서 생성 중... =====\n")

    # 기본 확인
    if not state.get("conversation") and not state.get("evaluation"):
        print("아직 진행된 인터뷰가 없습니다.")
        return state

    evals = state.get("evaluation", [])
    if not isinstance(evals, list) or len(evals) == 0:
        print("evaluation 항목이 비어있거나 형식이 올바르지 않습니다.")
        return state

    # question_strategy별로 그룹화
    groups = defaultdict(list)
    order = []  # 전략 등장 순서 보존용
    for item in evals:
        strat = item.get("question_strategy") or "기타"
        if strat not in groups:
            order.append(strat)
        groups[strat].append(item)

    # 모델 선언
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

    # 전략별 피드백용 프롬프트
    strategy_prompt = PromptTemplate(
        input_variables=["temp"],
        template=
        """
        당신은 인사담당 면접관입니다.
        아래의 질문 전략별 문답에 대한 총평을 작성하세요.
        출력 형식은 반드시 아래의 구조를 따르세요.
        불필요한 문장이나 서론 없이 형식 그대로 출력하십시오.

        [피드백 형식]
        [전략명]
        1. 답변 스타일: (지원자의 전반적인 답변 방식 요약)
        2. 강점: (잘한 점을 구체적으로 기술)
        3. 약점: (부족한 점 및 개선 포인트)
        4. 종합 평가: (해당 전략 영역에 대한 총평, 3~5문장 내외)

        --- 실제 입력 데이터 ---
        {temp}
        """
    )

    report_text = "===== 인터뷰 피드백 보고서 =====\n"
    strategy_feedbacks = []

    # 전략별 처리
    for strat in order:
        temp = ""
        items = groups[strat]

        report_text += f"\n--- 전략: {strat} (문항 수: {len(items)}) ---\n\n"

        for i, qna in enumerate(items, 1):
            question = (qna.get("question") or "").strip()
            answer = (qna.get("answer") or "").strip()
            evaluation = qna.get("evaluation", {})

            report_text += f"Q{i}. {question}\n"
            report_text += f"A. {answer}\n"

            if isinstance(evaluation, dict) and evaluation:
                report_text += "평가:\n"
                for k, v in evaluation.items():
                    report_text += f" - {k}: {v}\n"
            else:
                report_text += "평가: 없음\n"

            report_text += "\n"

            temp += f"질문: {question}\n답변: {answer}\n총평: {evaluation.get('총평','없음')}\n질문 전략: {strat}\n\n"

        # LLM을 통한 전략별 총평 생성
        formatted_prompt = strategy_prompt.format(temp=temp)
        raw_text = llm.invoke(formatted_prompt).content.strip()

        strategy_feedbacks.append(raw_text)
        report_text += "" + raw_text + "\n\n"

    # --- 전체 면접 총평 ---
    total_prompt = PromptTemplate(
        input_variables=["temp"],
        template=
        """
        당신은 인사담당 면접관입니다.
        아래는 면접에서 평가된 각 질문 전략별 피드백입니다.
        이 내용을 바탕으로 지원자에 대한 **면접 총평**을 작성하세요.
        출력 형식은 반드시 아래의 구조를 따르며, 불필요한 서론 없이 형식 그대로 출력하십시오.

        [출력 형식]
        [면접 총평]
        1. 전반적인 인상: (면접 중 지원자의 태도, 표현력, 커뮤니케이션 등 전반적 인상 요약)
        2. 강점 요약: (각 전략에서 반복적으로 드러난 강점 및 긍정적 특징)
        3. 개선점 요약: (보완이 필요한 부분, 구체적 개선 방향)
        4. 종합 평가: (지원자의 역량 수준 및 조직 적합성에 대한 총괄 의견 — 5문장 이내)

        --- 참고 데이터 (전략별 피드백) ---
        {temp}
        """
    )

    temp = "\n".join(strategy_feedbacks)
    formatted_prompt = total_prompt.format(temp=temp)
    overall_feedback = llm.invoke(formatted_prompt).content.strip()

    # 누적 출력에 총평 추가
    report_text += "\n" + overall_feedback + "\n"

    print(report_text)

    # state에 결과 저장
    state["strategy_feedbacks"] = strategy_feedbacks
    state["overall_feedback"] = overall_feedback
    state["report_text"] = report_text

    return state


# 6) Agent --------------------
def select_strategy_node(state: InterviewState) -> InterviewState:
    # 전략 목록 가져오기
    available_categories = list(state.get("question_strategy", {}).keys())

    # 사용된 전략 목록
    used_strategy = state.get("used_strategy", [])

    # 제외 후 남은 전략 목록
    remaining_categories = [s for s in available_categories if s not in used_strategy]
    
    # 남은 전략 중 랜덤 선택
    selected_strategy = random.choice(remaining_categories)
    selected_section = state["question_strategy"].get(selected_strategy, {})
    example_questions = selected_section.get("예시 질문", [])
    selected_question = random.choice(example_questions) if example_questions else f"{selected_strategy} 관련 질문을 생성하지 못했습니다."

    # 상태 업데이트
    state["current_question"] = selected_question
    state["current_strategy"] = selected_strategy
    state["current_answer"] = ""
    state.setdefault("used_strategy", []).append(selected_strategy)

    # print(f"[선택된 전략] {selected_strategy}")
    # print(f"[선택된 질문] {selected_question}")
    
    return state


# 분기 판단 함수
# 분기 판단 함수
def route_next(state: InterviewState) -> Literal["generate", "summarize", "select_strategy"]:
    next_step = state.get("next_step", "end")
    if next_step == "end":
        return "summarize"
    elif next_step == "additional_question":
        return "generate"
    else:
        return "select_strategy"

            
#그래프 생성
builder = StateGraph(InterviewState)

#노드 생성
builder.add_node("evaluate", evaluate_answer)
builder.add_node("decide", decide_next_step)
builder.add_node("generate", generate_question)
builder.add_node("summarize", summarize_interview)
builder.add_node("reflect", reflect_on_evaluation)
builder.add_node("re_evaluate", re_evaluate_answer)
builder.add_node("select_strategy", select_strategy_node)

# edge 연결
builder.add_edge(START,"evaluate")
builder.add_edge("evaluate", "reflect")
builder.add_edge("re_evaluate", "reflect")
builder.add_edge("select_strategy", END)
builder.add_conditional_edges(
    "re_evaluate",
    lambda state: state.get("next_step", "reflect"),
    {
        "reflect": "reflect",
        "decide_next_step": "decide",
    },
)
builder.add_conditional_edges(
    "reflect",
    lambda state: state.get("next_step", "re_evaluate"),
    {
        "re_evaluate": "re_evaluate",
        "decide_next_step": "decide"
    },
)
builder.add_conditional_edges(
    "decide",
    route_next,
    {
        "generate": "generate",
        "summarize": "summarize",
        "select_strategy": "select_strategy" 
    },
)

graph = builder.compile()
#-------------------------------------------------------------------


########### 다음 코드는 제공되는 gradio 코드 입니다.################

import gradio as gr
import tempfile

# 세션 상태 초기화 함수
def initialize_state():
    return {
        "state": None,
        "interview_started": False,
        "interview_ended": False,
        "chat_history": []
    }

# 파일 업로드 후 인터뷰 초기화
def upload_and_initialize(file_obj, session_state):
    if file_obj is None:
        return session_state, "파일을 업로드해주세요."

    # Gradio는 file_obj.name 이 파일 경로야
    file_path = file_obj.name

    # 인터뷰 사전 처리
    state = preProcessing_Interview(file_path)
    session_state["state"] = state
    session_state["interview_started"] = True

    # 첫 질문 저장
    first_question = state["current_question"]
    session_state["chat_history"].append(["🤖 AI 면접관", first_question])

    return session_state, session_state["chat_history"]

# 답변 처리 및 다음 질문 생성
def chat_interview(user_input, session_state):
    if not session_state["interview_started"]:
        return session_state, "먼저 이력서를 업로드하고 인터뷰를 시작하세요."

    # (1) 사용자 답변 저장
    session_state["chat_history"].append(["🙋‍♂️ 지원자", user_input])
    session_state["state"] = update_current_answer(session_state["state"], user_input)

    # (2) Agent 실행 (평가 및 다음 질문 or 종료)
    session_state["state"] = graph.invoke(session_state["state"])

    # (3) 종료 여부 판단
    if session_state["state"]["next_step"] == "end":
        session_state["interview_ended"] = True
        final_summary = "✅ 인터뷰가 종료되었습니다!\n\n"
        for i, turn in enumerate(session_state["state"]["conversation"]):
            final_summary += f"\n**[질문 {i+1}]** {turn['question']}\n**[답변 {i+1}]** {turn['answer']}\n"
            if i < len(session_state["state"]["evaluation"]):
                eval_result = session_state["state"]["evaluation"][i]
                final_summary += f"_평가 - 질문 연관성: {eval_result['질문과의 연관성']}, 답변 구체성: {eval_result['답변의 구체성']}_\n"
        session_state["chat_history"].append(["🤖 AI 면접관", final_summary])
        return session_state, session_state["chat_history"], gr.update(value="")

    else:
        next_question = session_state["state"]["current_question"]
        session_state["chat_history"].append(["🤖 AI 면접관", next_question])
        return session_state, session_state["chat_history"], gr.update(value="")

# Gradio 인터페이스 구성
with gr.Blocks() as demo:
    session_state = gr.State(initialize_state())

    gr.Markdown("# 🤖 AI 면접관 \n이력서를 업로드하고 인터뷰를 시작하세요!")

    with gr.Row():
        file_input = gr.File(label="이력서 업로드 (PDF 또는 DOCX)")
        upload_btn = gr.Button("인터뷰 시작")

    chatbot = gr.Chatbot()
    user_input = gr.Textbox(show_label=False, placeholder="답변을 입력하고 Enter를 누르세요.")

    upload_btn.click(upload_and_initialize, inputs=[file_input, session_state], outputs=[session_state, chatbot])
    user_input.submit(chat_interview, inputs=[user_input, session_state], outputs=[session_state, chatbot])
    user_input.submit(lambda: "", None, user_input)

# 실행
demo.launch(share=True)

Overwriting app.py


## **3. 실행**

In [4]:
!python app.py

^C
